# **Импорты**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import os
from tensorflow.keras.preprocessing import image
import gdown
import pandas as pd

# **Проверка GPU**

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

Используется устройство: cuda


# **Загрузка и распаковка датасета**

In [3]:
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l3/hw_light.zip', None, quiet=True)
!unzip -q hw_light.zip

# **Загрузка изображений**

In [4]:
base_dir = '/content/hw_light'
x_train = []
y_train = []
img_height, img_width = 20, 20

for patch in os.listdir(base_dir):
    for img in os.listdir(os.path.join(base_dir, patch)):
        img_path = os.path.join(base_dir, patch, img)
        img_array = image.img_to_array(
            image.load_img(img_path, target_size=(img_height, img_width), color_mode='grayscale')
        )
        x_train.append(img_array)
        # метки: 0 для папки '0', 1 для папки '3', 2 для остальных ('8')
        if patch == '0':
            y_train.append(0)
        elif patch == '3':
            y_train.append(1)
        else:
            y_train.append(2)

x_train = np.array(x_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.int64)

print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}")

x_train shape: (302, 20, 20, 1), y_train shape: (302,)


# **Предобработка**

In [5]:
x_train = x_train / 255.0
x_train_flat = x_train.reshape(x_train.shape[0], -1)

# **Преобразуем в тензоры PyTorch**

In [6]:
X_tensor = torch.tensor(x_train_flat, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.long)

# **Определение модели**

In [7]:
class SimpleNet(nn.Module):
    def __init__(self, hidden_size, activation='relu'):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(400, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 3)  # 3 класса
        self.activation = activation
        if activation == 'relu':
            self.act = nn.ReLU()
        else:
            self.act = nn.Identity()  # linear = отсутствие нелинейности

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x  # без softmax, он в loss

# **Параметры для перебора**

In [8]:
neurons_list = [10, 100, 5000]
activations = ['relu', 'linear']
batch_sizes = [10, 100, 1000]
epochs = 10
results = []

# **Цикл по всем комбинациям**

In [11]:
for neurons in neurons_list:
    for activation in activations:
        for batch_size in batch_sizes:
            print(f"\n--- Эксперимент: нейронов={neurons}, активация={activation}, batch_size={batch_size} ---")

            # Создаём даталоадер
            dataset = TensorDataset(X_tensor, y_tensor)
            dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

            # Модель, оптимизатор, функция потерь
            model = SimpleNet(neurons, activation).to(device)
            criterion = nn.CrossEntropyLoss()
            optimizer = optim.Adam(model.parameters(), lr=0.001)

            # Обучение
            model.train()
            for epoch in range(epochs):
                total_loss = 0
                for X_batch, y_batch in dataloader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    optimizer.zero_grad()
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()
            weights_filename = f"lite_{neurons}_{activation}_{batch_size}.weights.pt"
            torch.save(model.state_dict(), weights_filename)
            print(f"✅ Веса сохранены: {weights_filename}")

            # Оценка точности на всей обучающей выборке
            model.eval()
            with torch.no_grad():
                X_all = X_tensor.to(device)
                y_all = y_tensor.to(device)
                outputs = model(X_all)
                _, predicted = torch.max(outputs, 1)
                accuracy = (predicted == y_all).float().mean().item()

            print(f"Точность: {accuracy:.4f}")
            results.append({
                'neurons': neurons,
                'activation': activation,
                'batch_size': batch_size,
                'accuracy': accuracy
            })


--- Эксперимент: нейронов=10, активация=relu, batch_size=10 ---
✅ Веса сохранены: lite_10_relu_10.weights.pt
Точность: 0.7947

--- Эксперимент: нейронов=10, активация=relu, batch_size=100 ---
✅ Веса сохранены: lite_10_relu_100.weights.pt
Точность: 0.7020

--- Эксперимент: нейронов=10, активация=relu, batch_size=1000 ---
✅ Веса сохранены: lite_10_relu_1000.weights.pt
Точность: 0.6457

--- Эксперимент: нейронов=10, активация=linear, batch_size=10 ---
✅ Веса сохранены: lite_10_linear_10.weights.pt
Точность: 0.7583

--- Эксперимент: нейронов=10, активация=linear, batch_size=100 ---
✅ Веса сохранены: lite_10_linear_100.weights.pt
Точность: 0.5828

--- Эксперимент: нейронов=10, активация=linear, batch_size=1000 ---
✅ Веса сохранены: lite_10_linear_1000.weights.pt
Точность: 0.6490

--- Эксперимент: нейронов=100, активация=relu, batch_size=10 ---
✅ Веса сохранены: lite_100_relu_10.weights.pt
Точность: 0.8245

--- Эксперимент: нейронов=100, активация=relu, batch_size=100 ---
✅ Веса сохранены: 

# **Таблица результатов**

In [10]:
df = pd.DataFrame(results)
df = df.sort_values(by=['neurons', 'activation', 'batch_size'])
print("\n             Таблица результатов")
print(df.to_string(index=False))


             Таблица результатов
 neurons activation  batch_size  accuracy
      10     linear          10  0.850993
      10     linear         100  0.486755
      10     linear        1000  0.692053
      10       relu          10  0.758278
      10       relu         100  0.649007
      10       relu        1000  0.423841
     100     linear          10  0.870861
     100     linear         100  0.615894
     100     linear        1000  0.768212
     100       relu          10  0.887417
     100       relu         100  0.708609
     100       relu        1000  0.768212
    5000     linear          10  0.834437
    5000     linear         100  0.662252
    5000     linear        1000  0.543046
    5000       relu          10  0.983444
    5000       relu         100  0.748344
    5000       relu        1000  0.615894
